# Recommendation System

## Content Based Filtering

In [1]:
import numpy as np

# random dataset
books = [
    {"title": "The Hobbit", "genre": "Fantasy", "author": "Tolkien"},
    {"title": "Harry Potter", "genre": "Fantasy", "author": "Rowling"},
    {"title": "1984", "genre": "Dystopian", "author": "Orwell"},
    {"title": "Animal Farm", "genre": "Dystopian", "author": "Orwell"},
    {"title": "The Silmarillion", "genre": "Fantasy", "author": "Tolkien"}
]

# Build feature vectors manually
genres = sorted(list(set(b["genre"] for b in books)))
authors = sorted(list(set(b["author"] for b in books)))

In [2]:
def vectorize_book(book):
    vec = []
    for g in genres:
        vec.append(1 if book["genre"]==g else 0)
    for a in authors:
        vec.append(1 if book["author"]==a else 0)
    return np.array(vec)

book_vectors = np.array([vectorize_book(b) for b in books])


In [3]:
# Cosine similarity
def cosine_sim(vec1, vec2):
    if np.linalg.norm(vec1)==0 or np.linalg.norm(vec2)==0:
        return 0
    return np.dot(vec1, vec2)/(np.linalg.norm(vec1)*np.linalg.norm(vec2))

In [4]:
# Recommend top-N similar books
def recommend_books(book_index, N=2):
    sims = [cosine_sim(book_vectors[book_index], v) for v in book_vectors]
    sims[book_index] = -1  # exclude the same book
    top_idx = np.argsort(sims)[-N:][::-1]
    return [books[i]["title"] for i in top_idx]

print("Content-Based Recommendations for 'The Hobbit':", recommend_books(0))

Content-Based Recommendations for 'The Hobbit': ['The Silmarillion', 'Harry Potter']


## Collaborative Filtering

#### User-Movie ratings

In [5]:
# user-movie ratings
movies = ["Inception", "Interstellar", "The Matrix", "Titanic"]
users = ["Alice", "Bob", "Charlie", "David"]

ratings = np.array([
    [5, 4, 5, 1],  # Alice
    [4, 5, 5, 2],  # Bob
    [1, 2, 1, 5],  # Charlie
    [2, 1, 2, 5]   # David
])


In [6]:
# Cosine similarity between users
def user_similarity(user_idx):
    sims = []
    u1 = ratings[user_idx]
    for i in range(ratings.shape[0]):
        if i != user_idx:
            u2 = ratings[i]
            sim = cosine_sim(u1, u2)
            sims.append(sim)
        else:
            sims.append(-1)
    return sims

In [7]:
# Predict ratings for a user (weighted average)
def predict_ratings(user_idx):
    sims = user_similarity(user_idx)
    pred = np.zeros(ratings.shape[1])
    for j in range(ratings.shape[1]):
        numer = sum(sims[i]*ratings[i,j] for i in range(ratings.shape[0]) if sims[i]>0)
        denom = sum(abs(sims[i]) for i in range(ratings.shape[0]) if sims[i]>0)
        pred[j] = numer/denom if denom!=0 else 0
    return pred

In [8]:
# Recommend top-N movies for Alice
user_idx = 0
pred_ratings = predict_ratings(user_idx)
top_movies = [movies[i] for i in np.argsort(pred_ratings)[::-1] if i not in [0,1,2,3]][:2]  # avoid already rated? Here just demo
print("Collaborative Filtering Predicted Ratings for Alice:", pred_ratings)

Collaborative Filtering Predicted Ratings for Alice: [2.69453336 3.11326419 3.16249962 3.59610123]
